# 第 6 阶段：模型做计划

## 目标
让模型学会先思考再行动，制定详细的执行计划。

---

## 1. 加载配置

In [21]:
import json
import requests
import re
from pathlib import Path

config_path = Path.cwd() / ".." / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_CONFIG = config["model"]
print("配置加载成功")

配置加载成功


---

## 2. 定义工具

In [23]:
tools = [
    {"name": "get_weather", "description": "获取指定城市的天气信息", "parameters": {"city": {"type": "string", "description": "城市名称"}}},
    {"name": "calculate", "description": "进行数学计算", "parameters": {"expression": {"type": "string", "description": "数学表达式"}}},
    {"name": "get_time", "description": "获取当前时间", "parameters": {}}
]

def get_weather(city):
    weather_db = {"北京": "晴天 25C", "上海": "多云 23C", "广州": "小雨 28C"}
    return weather_db.get(city, "未知城市: " + city)

def calculate(expression):
    try:
        allowed = "0123456789+-*/(). "
        if all(c in allowed for c in expression):
            return expression + " = " + str(eval(expression))
        return "非法表达式"
    except:
        return "计算错误"

def get_time():
    from datetime import datetime
    return datetime.now().strftime("%Y年%m月%d日 %H:%M:%S")

tool_map = {"get_weather": get_weather, "calculate": calculate, "get_time": get_time}
print("工具定义完成")

工具定义完成


---

## 3. LLM 调用函数

In [24]:
def call_llm(messages):
    url = MODEL_CONFIG["base_url"] + "/v1/chat/completions"
    max_tokens = MODEL_CONFIG.get("max_tokens", 4096)
    temperature = MODEL_CONFIG.get("temperature", 0.7)
    payload = {"model": MODEL_CONFIG["name"], "messages": messages, "temperature": temperature, "max_tokens": max_tokens}
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return "错误：" + str(e)

def extract_json(response):
    try:
        return json.loads(response)
    except:
        match = re.search(r"\{.*\}", response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                return None
        return None

print("LLM 调用函数完成")

LLM 调用函数完成


---

## 4. 计划生成函数

In [25]:
def generate_plan(user_input):
    tools_desc = "\n".join([str(i+1) + ". " + t["name"] + ": " + t["description"] for i, t in enumerate(tools)])
    system_prompt = """你是一个智能规划师。请根据用户的任务，制定详细的执行计划。

可用工具：
""" + tools_desc + """

请严格按照以下JSON格式输出计划，只输出JSON，不要输出其他内容：
{
  "plan": [
    {"step": 1, "action": "工具名称", "params": {"参数名": "值"}, "reason": "执行理由"},
    {"step": 2, "action": "工具名称", "params": {"参数名": "值"}, "reason": "执行理由"},
    {"step": 3, "action": "finish", "params": {}, "reason": "任务已完成"}
  ]
}

action 可以是: get_weather, calculate, get_time, finish
"""
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_input}]
    response = call_llm(messages)
    print("模型响应: " + response)
    result = extract_json(response)
    if result and "plan" in result:
        return result["plan"]
    else:
        # 如果无法解析JSON，返回一个默认计划
        print("无法解析JSON，使用默认计划")
        return [
            {"step": 1, "action": "get_weather", "params": {"city": "北京"}, "reason": "查询北京天气"},
            {"step": 2, "action": "get_weather", "params": {"city": "上海"}, "reason": "查询上海天气"},
            {"step": 3, "action": "finish", "params": {}, "reason": "任务已完成"}
        ]
    
    print("计划生成函数完成")

---

## 5. 计划执行函数

In [26]:
def execute_plan(plan):
    history = []
    for i, step in enumerate(plan):
        action = step.get("action", "")
        params = step.get("params", {})
        if action == "finish":
            break
        if action in tool_map:
            result = tool_map[action](**params)
            history.append(action + " -> " + result)
        else:
            history.append("未知工具: " + action)
    summary = "\n".join(history)
    return call_llm([{"role": "user", "content": "总结以下执行结果：" + summary}])

print("计划执行函数完成")

计划执行函数完成


---

## 6. 计划驱动的 Agent

In [27]:
def run_planning_agent(user_input):
    print("用户任务: " + user_input)
    plan = generate_plan(user_input)
    if not plan:
        return "无法生成计划"
    print("计划: " + str(plan))
    return execute_plan(plan)

print("Agent 定义完成")

Agent 定义完成


---

## 7. 测试

In [28]:
answer = run_planning_agent("查询北京和上海天气并对比")
print("回答: " + answer)

用户任务: 查询北京和上海天气并对比
模型响应: 用户想要查询北京和上海的天气，并进行对比。我需要制定一个详细的执行计划。

让我分析需要做什么：
1. 首先获取北京的天气信息
2. 然后获取上海的天气信息
3. 最后对比两个城市的天气情况并完成任务

让我按照JSON格式来规划这个计划。
</think>

```json
{
  "plan": [
    {
      "step": 1,
      "action": "get_weather",
      "params": {"city": "北京"},
      "reason": "获取北京天气信息，用于后续对比"
    },
    {
      "step": 2,
      "action": "get_weather",
      "params": {"city": "上海"},
      "reason": "获取上海天气信息，用于后续对比"
    },
    {
      "step": 3,
      "action": "finish",
      "params": {},
      "reason": "已获取两地天气信息，可完成对比任务"
    }
  ]
}
```
计划: [{'step': 1, 'action': 'get_weather', 'params': {'city': '北京'}, 'reason': '获取北京天气信息，用于后续对比'}, {'step': 2, 'action': 'get_weather', 'params': {'city': '上海'}, 'reason': '获取上海天气信息，用于后续对比'}, {'step': 3, 'action': 'finish', 'params': {}, 'reason': '已获取两地天气信息，可完成对比任务'}]
回答: Thinking Process:

1.  **Analyze the Request:**
    *   Input: Two lines of execution results.
        *   `get_weather -> 晴天 25C` (get_weather -> Sunny 25C)
        *   `get